# 03 — Attribute Library

In [1]:
# Cell 1 — setup
import sys, pathlib
sys.path.append(str(pathlib.Path.cwd().parent / "src"))
from scorelab import config, data
con = data.connect()            # read-only DuckDB connection to data/lc.duckdb

In [2]:
# Cell 2 — sanity check
print(con.sql("SHOW TABLES"))
print(con.sql("SELECT COUNT(*) AS n, MIN(issue_d), MAX(issue_d) FROM accepted"))

┌─────────────────┐
│      name       │
│     varchar     │
├─────────────────┤
│ accepted        │
│ rejected        │
│ v_accepted_asof │
│ v_split         │
│ v_target        │
└─────────────────┘

┌─────────┬──────────────┬──────────────┐
│    n    │ min(issue_d) │ max(issue_d) │
│  int64  │     date     │     date     │
├─────────┼──────────────┼──────────────┤
│ 2260668 │ 2007-06-01   │ 2018-12-01   │
└─────────┴──────────────┴──────────────┘



In [3]:
# Cell 3 — common imports and settings for this notebook
import json, warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.width", 180); pd.set_option("display.max_columns", 40); pd.set_option("display.max_rows", 200)
from scorelab import target, attributes as A, binning as B, scorecard as S, validation as V, fairness as F
from scorelab import challengers as C, reject_inference as R, monitoring as M, report as REP
MODELS = config.MODELS_DIR; MODELS.mkdir(exist_ok=True)
SAMPLE_FRAC = None     # None = full data (recommended for the fitting notebooks); e.g. 0.10 for a quick pass
cols = [r[0] for r in con.execute("DESCRIBE v_target").fetchall()]
assert "label" in cols, "v_target is the old stub: close this kernel and run `python scripts/refresh_views.py`"
def load_frame(splits=("train", "validate", "oot"), **kw):
    """Observable, non-excluded loans with attributes built; indeterminates dropped unless asked."""
    f = target.modeling_frame(con, splits=splits, sample_frac=SAMPLE_FRAC, **kw)
    return A.build_attributes(f)
MEANINGS = {a.name: a.meaning for a in A.REGISTRY}
DIRECTIONS = {a.name: a.direction for a in A.REGISTRY}

## What this notebook settles

The candidate attribute list that goes to classing. The registry in `scorelab.attributes` holds every raw bureau field the keep-list allows plus a handful of derived ratios, each with a family, a plain-language meaning (used verbatim in reason codes) and an expected direction of risk (used as the monotone constraint). You decide which enter, and you own the meanings.

Decisions: **C — attribute families and exclusions**, **D — derived attributes to keep or add**, **E — the candidate list** written to `models/candidate_attributes.json`.

In [4]:
# Leakage control: every source column of every attribute must be a keep-list input (raises otherwise)
A.leakage_check()
reg = A.registry_table(); display(reg)
reg.to_csv(config.DOCS_DIR / "exhibits" / "attribute_registry.csv", index=False)

,attribute,family,meaning,direction,missing_means,derived,available_from,sources
0,inq_last_6mths,inquiries,Hard inquiries in the last 6 months,riskier,no inquiries recorded,False,2012,inq_last_6mths
1,inq_last_12m,inquiries,Hard inquiries in the last 12 months,riskier,field not reported,False,2016,inq_last_12m
2,mths_since_recent_inq,inquiries,Months since the most recent inquiry,safer,never inquired,False,2012,mths_since_recent_inq
3,delinq_2yrs,delinquency,30+ DPD incidences in the last 2 years,riskier,none,False,2012,delinq_2yrs
4,mths_since_last_delinq,delinquency,Months since the last delinquency,safer,never delinquent,False,2012,mths_since_last_delinq
5,mths_since_last_major_derog,delinquency,Months since the last 90+ DPD or worse,safer,never 90+,False,2012,mths_since_last_major_derog
6,mths_since_recent_bc_dlq,delinquency,Months since the last bankcard delinquency,safer,never delinquent on a bankcard,False,2012,mths_since_recent_bc_dlq
7,mths_since_recent_revol_delinq,delinquency,Months since the last revolving delinquency,safer,never delinquent on revolving,False,2012,mths_since_recent_revol_delinq
8,num_tl_90g_dpd_24m,delinquency,Trades 90+ DPD in the last 24 months,riskier,none,False,2012,num_tl_90g_dpd_24m
9,num_tl_30dpd,delinquency,Trades currently 30 DPD,riskier,none,False,2012,num_tl_30dpd


In [5]:
# Build the attributes on the modelling frame
frame = load_frame()
tr = frame[frame.split == "train"]
names = A.attribute_names()
print(frame.shape, "train", tr.shape, "attributes", len(names))
gaps = json.loads((MODELS / "audit_decisions.json").read_text(encoding="utf-8"))["structural_gap_fields"] if (MODELS / "audit_decisions.json").exists() else []
print("structural-gap fields from NB01:", gaps)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(1198176, 84) train (364774, 84) attributes 59
structural-gap fields from NB01: ['all_util', 'il_util', 'inq_last_12m', 'max_bal_bc', 'mths_since_rcnt_il', 'open_il_24m', 'open_rv_24m']


In [6]:
# Evidence — availability by vintage (attributes, after derivation)
display(A.availability_by_vintage(frame, names))

vintage_year,2013,2014,2015,2016
inq_last_6mths,1.000,1.000,1.000,1.000
inq_last_12m,0.000,0.000,0.050,1.000
mths_since_recent_inq,0.891,0.908,0.893,0.894
delinq_2yrs,1.000,1.000,1.000,1.000
mths_since_last_delinq,0.435,0.508,0.515,0.528
mths_since_last_major_derog,0.201,0.282,0.291,0.289
mths_since_recent_bc_dlq,0.223,0.264,0.258,0.256
mths_since_recent_revol_delinq,0.301,0.360,0.360,0.363
num_tl_90g_dpd_24m,1.000,1.000,1.000,1.000
num_tl_30dpd,1.000,1.000,1.000,1.000


In [7]:
# Evidence — univariate rank power on train, with the expected vs empirical direction
screen = A.univariate_screen(tr, names)
display(screen)
screen.to_csv(config.DOCS_DIR / "exhibits" / "univariate_screen_train.csv", index=False)
print("direction conflicts (expected vs empirical):")
display(screen[(screen.expected_direction != "no prior") & (screen.expected_direction != screen.empirical_direction)])

,attribute,family,missing_rate,n_distinct,auc_higher_is_riskier,gini_abs,expected_direction,empirical_direction
0,new_trades_share_24m,tradelines,0.0000,959,0.5936,0.1873,riskier,riskier
1,acc_open_past_24mths,tradelines,0.0000,37,0.5809,0.1619,riskier,riskier
2,num_tl_op_past_12m,tradelines,0.0000,23,0.5728,0.1457,riskier,riskier
3,bal_ex_mort_to_limit,balances_utilisation,0.0000,363469,0.5658,0.1315,riskier,riskier
4,bc_open_to_buy,balances_utilisation,0.0096,45201,0.4356,0.1289,safer,safer
5,mo_sin_rcnt_tl,credit_age,0.0000,140,0.4383,0.1234,safer,safer
6,tot_hi_cred_lim,balances_utilisation,0.0000,214763,0.4402,0.1197,safer,safer
7,total_bc_limit,balances_utilisation,0.0000,9370,0.4417,0.1166,safer,safer
8,avg_cur_bal,balances_utilisation,0.0000,54786,0.4445,0.1110,safer,safer
9,mo_sin_old_rev_tl_op,credit_age,0.0000,690,0.4450,0.1101,safer,safer


direction conflicts (expected vs empirical):


,attribute,family,missing_rate,n_distinct,auc_higher_is_riskier,gini_abs,expected_direction,empirical_direction
24,mths_since_last_record,derogatory,0.8450,122,0.5170,0.0340,safer,riskier
27,num_sats,tradelines,0.0000,62,0.5080,0.0159,safer,riskier
37,pct_tl_nvr_dlq,delinquency,0.0004,471,0.5040,0.0080,safer,riskier
46,mths_since_last_major_derog,delinquency,0.7477,152,0.5007,0.0013,safer,riskier
49,tax_liens,derogatory,0.0000,24,0.4999,0.0002,riskier,safer
50,chargeoff_within_12_mths,derogatory,0.0000,8,0.4999,0.0002,riskier,safer
51,num_tl_120dpd_2m,delinquency,0.0219,3,0.5000,0.0001,riskier,safer
52,inq_last_12m,inquiries,1.0000,0,NaN,NaN,riskier,
53,open_rv_24m,tradelines,1.0000,0,NaN,NaN,riskier,
54,open_il_24m,tradelines,1.0000,0,NaN,NaN,riskier,


In [8]:
# Evidence — delinquency check: built attributes against the raw fields, and bad rate by bucket on train
DQ = ["delinq_2yrs", "mths_since_last_delinq", "pub_rec", "inq_last_6mths"]
raw = target.modeling_frame(con, splits=("train",), columns=["id"] + DQ)
chk = tr[["id"] + DQ].merge(raw[["id"] + DQ], on="id", suffixes=("_built", "_raw"))
for c in DQ:
    b, r = chk[c + "_built"], chk[c + "_raw"]
    same = ((b == r) | (b.isna() & r.isna())).mean()
    print(f"{c}: {len(chk):,} train rows · built equals raw on {same:.2%} · raw missing {r.isna().mean():.2%}"
          f" · raw negative {int((r < 0).sum())} · raw range {r.min():g} to {r.max():g}")
BUCKETS = {"delinq_2yrs": [0, 1, 2], "mths_since_last_delinq": [6, 12, 24, 48], "pub_rec": [0, 1], "inq_last_6mths": [0, 1, 2, 3]}
for c, edges in BUCKETS.items():
    display(A.bucket_table(tr, c, edges))

delinq_2yrs: 364,774 train rows · built equals raw on 100.00% · raw missing 0.00% · raw negative 0 · raw range 0 to 29
mths_since_last_delinq: 364,774 train rows · built equals raw on 100.00% · raw missing 51.87% · raw negative 0 · raw range 0 to 188
pub_rec: 364,774 train rows · built equals raw on 100.00% · raw missing 0.00% · raw negative 0 · raw range 0 to 63
inq_last_6mths: 364,774 train rows · built equals raw on 100.00% · raw missing 0.00% · raw negative 0 · raw range 0 to 6


,delinq_2yrs,n,bads,share,bad_rate
0,<= 0,294650,28070,0.8078,0.0953
1,"(0, 1]",46606,4456,0.1278,0.0956
2,"(1, 2]",13980,1469,0.0383,0.1051
3,> 2,9538,1010,0.0261,0.1059


,mths_since_last_delinq,n,bads,share,bad_rate
0,<= 6,13549,1466,0.0371,0.1082
1,"(6, 12]",21197,2084,0.0581,0.0983
2,"(12, 24]",37351,3560,0.1024,0.0953
3,"(24, 48]",58624,5409,0.1607,0.0923
4,> 48,44844,4373,0.1229,0.0975
5,Missing,189209,18113,0.5187,0.0957


,pub_rec,n,bads,share,bad_rate
0,<= 0,308216,29345,0.8450,0.0952
1,"(0, 1]",48507,4840,0.1330,0.0998
2,> 1,8051,820,0.0221,0.1019


,inq_last_6mths,n,bads,share,bad_rate
0,<= 0,192829,15718,0.5286,0.0815
1,"(0, 1]",102229,10438,0.2803,0.1021
2,"(1, 2]",43341,5245,0.1188,0.1210
3,"(2, 3]",17802,2379,0.0488,0.1336
4,> 3,8573,1225,0.0235,0.1429


In [9]:
# Evidence — redundancy: pairs of attributes with |Spearman| > 0.7 on train
corr = tr[names].corr(method="spearman")
pairs = (corr.where(np.triu(np.ones(corr.shape), 1).astype(bool)).stack().rename("rho").reset_index()
         .query("abs(rho) > 0.7").sort_values("rho", key=abs, ascending=False))
display(pairs)

,level_0,level_1,rho
1210,open_acc,num_sats,0.998320
2061,revol_util,revol_bal_to_limit,0.981469
2525,tot_cur_bal,tot_hi_cred_lim,0.979490
824,pub_rec,any_derog_flag,0.943896
2521,tot_cur_bal,avg_cur_bal,0.935347
2885,mo_sin_old_rev_tl_op,credit_age_months,0.899878
2584,avg_cur_bal,tot_hi_cred_lim,0.899865
361,mths_since_recent_bc_dlq,mths_since_recent_revol_delinq,0.884053
766,pct_tl_nvr_dlq,ever_delinquent_flag,-0.878354
243,mths_since_last_delinq,mths_since_recent_revol_delinq,0.874714


### Decision C — Families and exclusions

Which families carry the card (delinquency, inquiries, utilisation, credit age, file depth)? Anything to exclude for business or compliance reasons even if predictive? The structural-gap fields from NB01 are dropped by default.

**Your reasoning:**

**DECIDED 2026-10-31** (drafted by Claude from this notebook's full-data outputs, applied; pending final review by Siavash)

**The card draws on four families:** new-account activity (tradelines), balances and utilisation, credit age, and inquiries. These are the families with the strongest single attributes on 2013–2014:

| Attribute | Gini |
|---|---:|
| `new_trades_share_24m` | 0.1873 |
| `acc_open_past_24mths` | 0.1619 |
| `num_tl_op_past_12m` | 0.1457 |
| `bal_ex_mort_to_limit` | 0.1315 |
| `bc_open_to_buy` | 0.1287 |
| `mo_sin_rcnt_tl` | 0.1234 |

**Delinquency and derogatory attributes are weak on this population.**
- The best delinquency attribute is `mths_since_recent_bc_dlq`, at Gini 0.0146.
- `mths_since_last_delinq` is 0.0128, `pub_rec` 0.0074 and `delinq_2yrs` 0.0074, all below the 0.02 cut.
- So the card will carry no delinquency attribute, and its reason codes will never cite one. Record this as a limitation of a booked Lending Club population, not a property of bureau data.
- **The weak signal is real, not a coding error.** The delinquency check above compares each built attribute with the raw field for all 364,774 train loans: built equals raw on 100.00% for `delinq_2yrs`, `mths_since_last_delinq`, `pub_rec` and `inq_last_6mths`, with no negative values.
- **By bucket, the delinquency fields barely separate risk:**
  - `delinq_2yrs`: 9.53% bad at 0 against 10.59% at 3 or more.
  - `pub_rec`: 9.52% at 0 against 10.19% at 2 or more.
  - `mths_since_last_delinq`: 10.82% within 6 months against 9.57% when missing (never delinquent, 51.87% of train), with no clear order in between.
- **For contrast, `inq_last_6mths` separates clearly** on the same loans, from 8.15% bad at 0 to 14.29% at 4 or more.

**No business or compliance exclusions are proposed** (`EXCLUDE = []`). Every attribute passes the leakage check (bureau fields on the keep-list only).

**Direction conflicts (registry against empirical):**
- `mths_since_last_record` (Gini 0.0340, 84.5% missing) is the only one above the cut. It goes to NB04, where its IV decides.
- `num_sats` (0.0159), `pct_tl_nvr_dlq` (0.0080), `mths_since_last_major_derog` (0.0013), `tax_liens`, `chargeoff_within_12_mths` and `num_tl_120dpd_2m` (≤ 0.0002) all fall below the cut.

**Still open from NB02 (Decision 3):** this notebook does not measure loans with a null `last_pymnt_d` or thin states and purposes. State and purpose are not inputs, so this matters only for NB08's segment tables.

### Decision D — Derived attributes

The registry derives `credit_age_months`, `new_trades_share_24m`, `revol_bal_to_limit`, `bal_ex_mort_to_limit`, `any_derog_flag`, `ever_delinquent_flag`. Keep, drop, or add (edit `scorelab/attributes.py` and re-run)?

**Your reasoning:**

**DECIDED 2026-10-31** (drafted by Claude from this notebook's full-data outputs, applied; pending final review by Siavash)

**Keep all six derived attributes in the registry.**

- **`new_trades_share_24m` is the strongest attribute of all** (Gini 0.1873, against 0.1619 for its numerator `acc_open_past_24mths`; Spearman 0.75 between them).
- **`bal_ex_mort_to_limit`** is fourth (0.1315).
- **Two derived attributes duplicate a raw one**, and the stepwise correlation guard in NB05 will keep only one of each pair:
  - `credit_age_months` against `mo_sin_old_rev_tl_op` (Spearman 0.90; Gini 0.0949 against 0.1101);
  - `revol_bal_to_limit` against `revol_util` (Spearman 0.98; 0.0615 against 0.0626).
- **`any_derog_flag` (0.0087) and `ever_delinquent_flag`** fall below the Gini cut, so they do not reach classing.

**No new derived attributes are proposed** at this stage.

### Decision E — The candidate list

Default below: available from 2012, not a structural-gap field, single-variable Gini at least 0.02 on train, minus your exclusions. Edit `EXCLUDE` and the threshold, then run.

**Your reasoning:**

**DECIDED 2026-10-31** (drafted by Claude from this notebook's full-data outputs, applied; pending final review by Siavash)

**Keep the default rule:** available from 2013, not a structural-gap field, single-attribute Gini of at least 0.02 on train, and no exclusions.

**It gives 27 candidates:** `inq_last_6mths`, `mths_since_recent_inq`, `mths_since_last_record`, `total_acc`, `acc_open_past_24mths`, `num_tl_op_past_12m`, `num_bc_tl`, `num_rev_tl_bal_gt_0`, `mort_acc`, `revol_util`, `bc_util`, `percent_bc_gt_75`, `bc_open_to_buy`, `revol_bal`, `tot_cur_bal`, `avg_cur_bal`, `total_bc_limit`, `total_rev_hi_lim`, `tot_hi_cred_lim`, `mo_sin_old_rev_tl_op`, `mo_sin_old_il_acct`, `mo_sin_rcnt_rev_tl_op`, `mo_sin_rcnt_tl`, `credit_age_months`, `new_trades_share_24m`, `revol_bal_to_limit`, `bal_ex_mort_to_limit`.

**Utilisation fields are now correct.** This run uses the fixed percent cap. Missing utilisation now stays missing (`bc_util` 0.99% missing on train, `percent_bc_gt_75` 0.96%) instead of being set to 150%. The list is unchanged from before the fix, but the utilisation bins and screens below now describe the real data.

In [10]:
EXCLUDE = []                      # your business / compliance exclusions, by attribute name
MIN_GINI = 0.02
ok = screen[(screen.gini_abs >= MIN_GINI)].attribute.tolist()
candidates = [n for n in A.attribute_names(min_available_from=min(config.TRAIN_VINTAGES))
              if n in ok and n not in EXCLUDE and not set(A.REGISTRY_BY_NAME[n].sources) & set(gaps)]
(MODELS / "candidate_attributes.json").write_text(json.dumps({"candidates": candidates, "exclude": EXCLUDE, "min_gini": MIN_GINI}, indent=1), encoding="utf-8")
print(len(candidates), "candidates saved to", (MODELS / "candidate_attributes.json").relative_to(config.ROOT).as_posix()); print(candidates)

27 candidates saved to models/candidate_attributes.json
['inq_last_6mths', 'mths_since_recent_inq', 'mths_since_last_record', 'total_acc', 'acc_open_past_24mths', 'num_tl_op_past_12m', 'num_bc_tl', 'num_rev_tl_bal_gt_0', 'mort_acc', 'revol_util', 'bc_util', 'percent_bc_gt_75', 'bc_open_to_buy', 'revol_bal', 'tot_cur_bal', 'avg_cur_bal', 'total_bc_limit', 'total_rev_hi_lim', 'tot_hi_cred_lim', 'mo_sin_old_rev_tl_op', 'mo_sin_old_il_acct', 'mo_sin_rcnt_rev_tl_op', 'mo_sin_rcnt_tl', 'credit_age_months', 'new_trades_share_24m', 'revol_bal_to_limit', 'bal_ex_mort_to_limit']
